# Policy Gradients & Actor-Critic

**Topics:** Policy Gradient Theorem | Score Function Trick | REINFORCE | Actor-Critic | TD / Advantage

**Duration:** ~1 hour | **GPU:** Recommended (T4 sufficient)

## Setup & Imports

In [ ]:
!pip install tqdm gymnasium[classic-control] torch matplotlib numpy -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym, numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
torch.manual_seed(42); np.random.seed(42)

def smooth(x, w=20): return np.convolve(x, np.ones(w)/w, mode='valid')
def plot_rewards(r, title, w=20):
    plt.figure(figsize=(9,3)); plt.plot(r, alpha=0.3, label='raw')
    if len(r)>=w: plt.plot(range(w-1,len(r)), smooth(r,w), label=f'smoothed({w})')
    plt.xlabel('Episode'); plt.ylabel('Return'); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()

print('Setup complete')


---
# Part 1 - Policy Gradient Theorem & Score Function Trick

## Theory

We want a policy $\pi_\theta(a|s)$ that maximises expected return:
$$J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\!\left[\sum_t \gamma^t r_t\right]$$

The environment is a black box, so we cannot backprop through it. The **Policy Gradient Theorem** gives us:
$$\nabla_\theta J = \mathbb{E}\!\left[\sum_t \nabla_\theta \log\pi_\theta(a_t|s_t)\cdot G_t\right]$$

This follows from the **score function (log-derivative) trick**:
$$\nabla_\theta\,\mathbb{E}_{x\sim p_\theta}[f(x)] = \mathbb{E}\!\left[f(x)\cdot\nabla_\theta\log p_\theta(x)\right]$$

*Intuition*: scale the gradient of the log-prob by how good the outcome was.


In [ ]:
# Score function trick — toy demo
# Maximise E[f(x)] where x ~ N(mu,1), f(x) = -(x-3)^2  => optimum at mu=3

import re


def score_grad(mu, n=500):
    x = np.random.randn(n) + mu       # sample x ~ N(mu,1)
    f = -(x - 3)**2                   # reward (peaks at x=3)

    # TODO: compute the score function d/d_mu log N(x; mu, 1)
    # Hint: for N(mu,1), log p = -0.5*(x-mu)^2 + const  =>  d/d_mu log p = ?
    score = None   # <-- replace with correct expression

    # TODO: return the score-function gradient estimate  E[f * score]
    return None    # <-- replace

mu, lr = 0.0, 0.1
history = [mu]
for _ in range(100):
    mu += lr * score_grad(mu)
    history.append(mu)

plt.figure(figsize=(8,3))
plt.plot(history); plt.axhline(3, color='r', ls='--', label='Optimum mu=3')
plt.xlabel('Step'); plt.ylabel('mu')
plt.title('Score Function Trick converging to optimum')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Final mu: {history[-1]:.3f}  (target 3.0)')


### Extension: Baseline reduces variance

Adding any constant baseline $b$ leaves the gradient **unbiased** but reduces variance:
$$\nabla_\theta J \approx \mathbb{E}[(f(x)-b)\cdot\nabla_\theta\log p_\theta(x)]$$
The best baseline is $b\approx\mathbb{E}[f(x)]$.


In [ ]:
# Compare gradient variance with/without baseline
def grad_variance(mu, trials=200, n=50, baseline=False):
    grads = []
    for _ in range(trials):
        x = np.random.randn(n) + mu
        f = -(x-3)**2; score = x-mu

        # TODO: if baseline=True, set b = E[f]; else b = 0
        b = None   # <-- replace

        # TODO: compute one gradient estimate using (f - b) * score
        g = None   # <-- replace

        grads.append(g)
    return np.var(grads)

mu = 1.0
v0 = grad_variance(mu, baseline=False)
v1 = grad_variance(mu, baseline=True)
print(f'Variance without baseline: {v0:.2f}')
print(f'Variance with    baseline: {v1:.2f}  ({v0/v1:.1f}x reduction)')


---
# Part 2 - REINFORCE

## Theory

REINFORCE (Williams 1992) is the simplest policy gradient algorithm:

1. Collect full episode under $\pi_\theta$
2. Compute returns $G_t = \sum_{t'\ge t}\gamma^{t'-t}r_{t'}$
3. Update: $\theta \leftarrow \theta + \alpha\sum_t \nabla_\theta\log\pi_\theta(a_t|s_t)\cdot G_t$

Loss (minimised): $\mathcal{L} = -\sum_t \log\pi_\theta(a_t|s_t)\cdot(G_t - b)$

Properties: ✓ Unbiased | ✗ High variance | ✗ Sample-inefficient


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_act, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.ReLU(),
            nn.Linear(h, h),       nn.ReLU(),
            nn.Linear(h, n_act)
        )
    def forward(self, x): return self.net(x)
    def act(self, s):
        s = torch.FloatTensor(s).unsqueeze(0).to(device)
        dist = Categorical(logits=self.forward(s))
        a = dist.sample()
        return a.item(), dist.log_prob(a)

def compute_returns(rewards, gamma=0.99):
    # TODO: compute discounted returns G_t = r_t + gamma*G_{t+1}
    # Hint: iterate in reverse so each G is built from the next
    G, returns = 0, []
    for r in reversed(rewards):
        # G = ?
        # returns.insert(0, G)
        pass   # <-- replace this block
    return returns

def reinforce_update(policy, opt, log_probs, rewards, gamma=0.99):
    returns = torch.FloatTensor(compute_returns(rewards, gamma)).to(device)

    # TODO: normalise returns: subtract mean, divide by std (+ 1e-8)
    returns = None   # <-- replace

    # TODO: compute loss = -mean(log_probs * returns)
    # Hint: torch.stack(log_probs) gives a tensor of log pi(a_t|s_t)
    loss = None   # <-- replace

    opt.zero_grad(); loss.backward(); opt.step()

def train_reinforce(n_ep=500):
    env = gym.make('CartPole-v1')
    policy = PolicyNet(4, 2).to(device)
    opt = optim.Adam(policy.parameters(), lr=3e-3)
    all_r = []
    for ep in tqdm(range(n_ep), desc='Training REINFORCE'):
        s, _ = env.reset(); lps, rs, done = [], [], False
        while not done:
            a, lp = policy.act(s)
            s, r, te, tr, _ = env.step(a); done = te or tr
            lps.append(lp); rs.append(r)
        reinforce_update(policy, opt, lps, rs)
        all_r.append(sum(rs))
        if (ep+1) % 100 == 0:
            print(f'Ep {ep+1:4d} | Mean-100: {np.mean(all_r[-100:]):.1f}')
    env.close(); return all_r, policy

reinforce_rewards, reinforce_policy = train_reinforce()
plot_rewards(reinforce_rewards, 'REINFORCE on CartPole-v1')


### Extension: REINFORCE with learned baseline

Instead of using the return $G_t$ directly, we can subtract a **state-dependent baseline** $V_\phi(s_t)$:
$$A_t = G_t - V_\phi(s_t)$$

This keeps the policy update focused on whether the outcome was **better or worse than expected** from that state.

We now train two networks:
- **Policy** $\pi_\theta(a|s)$ with REINFORCE using $A_t$
- **Value network** $V_\phi(s)$ by regressing onto the Monte Carlo returns $G_t$

Compare this against the return-normalisation trick above. Which one looks more stable?


In [ ]:
class ValueNet(nn.Module):
    def __init__(self, obs_dim, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.ReLU(),
            nn.Linear(h, h),       nn.ReLU(),
            nn.Linear(h, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def reinforce_baseline_update(policy, value_net, opt_policy, opt_value, states, log_probs, rewards, gamma=0.99):
    states  = torch.FloatTensor(np.array(states)).to(device)
    returns = torch.FloatTensor(compute_returns(rewards, gamma)).to(device)

    # TODO: predict V(s_t) for every visited state
    values = None  # <-- replace

    # TODO: compute advantages = returns - values
    advantages = None   # <-- replace

    # TODO: policy loss = -mean(log_probs * advantages.detach())
    policy_loss = None   # <-- replace

    # TODO: value loss = MSE(values, returns)
    value_loss = None   # <-- replace

    opt_policy.zero_grad()
    policy_loss.backward()
    opt_policy.step()

    opt_value.zero_grad()
    value_loss.backward()
    opt_value.step()

def train_reinforce_with_baseline(n_ep=500):
    env = gym.make('CartPole-v1')
    policy = PolicyNet(4, 2).to(device)
    value_net = ValueNet(4).to(device)
    opt_policy = optim.Adam(policy.parameters(), lr=3e-3)
    opt_value  = optim.Adam(value_net.parameters(), lr=3e-3)
    all_r = []
    for ep in tqdm(range(n_ep), desc='Training REINFORCE with baseline'):
        s, _ = env.reset(); states, lps, rs, done = [], [], [], False
        while not done:
            a, lp = policy.act(s)
            states.append(s)
            s, r, te, tr, _ = env.step(a); done = te or tr
            lps.append(lp); rs.append(r)
        reinforce_baseline_update(policy, value_net, opt_policy, opt_value, states, lps, rs)
        all_r.append(sum(rs))
        if (ep+1) % 100 == 0:
            print(f'Ep {ep+1:4d} | Mean-100: {np.mean(all_r[-100:]):.1f}')
    env.close(); return all_r, policy, value_net

reinforce_bl_rewards, reinforce_bl_policy, reinforce_bl_value = train_reinforce_with_baseline()
plot_rewards(reinforce_bl_rewards, 'REINFORCE with baseline on CartPole-v1')

In [ ]:
plt.figure(figsize=(9,3))
plt.plot(smooth(reinforce_rewards), label='REINFORCE + return normalisation', color='steelblue')
plt.plot(smooth(reinforce_bl_rewards), label='REINFORCE + learned baseline', color='darkgreen')
plt.axhline(495, color='gray', ls='--', alpha=0.5, label='Solved')
plt.xlabel('Episode'); plt.ylabel('Smoothed Return')
plt.title('REINFORCE variants on CartPole-v1')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Normalised REINFORCE final 50-ep mean: {np.mean(reinforce_rewards[-50:]):.1f}')
print(f'Baseline   REINFORCE final 50-ep mean: {np.mean(reinforce_bl_rewards[-50:]):.1f}')

---
# Part 3 - Actor-Critic & TD Error / Advantage

## Theory

**Advantage function:** $A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$  
How much better is action $a$ compared to the average?

**1-step TD error** as advantage estimate:
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**Actor-Critic** maintains two components:
- **Actor** $\pi_\theta(a|s)$ — acts in the environment
- **Critic** $V_\phi(s)$ — estimates state values

Updates:
$$\theta \leftarrow \theta + \alpha_\theta\,\nabla_\theta\log\pi_\theta(a_t|s_t)\cdot\delta_t$$
$$\phi \leftarrow \phi - \alpha_\phi\,\nabla_\phi\,(\delta_t)^2$$

Properties: ✓ Lower variance than REINFORCE | ✗ Introduces bias from approximate $V$


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, h=128):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(obs_dim, h), nn.ReLU())
        self.actor  = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, n_act))
        self.critic = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, x):
        f = self.shared(x)
        return self.actor(f), self.critic(f)
    def act(self, s):
        s = torch.FloatTensor(s).unsqueeze(0).to(device)
        logits, v = self.forward(s)
        dist = Categorical(logits=logits); a = dist.sample()
        return a.item(), dist.log_prob(a), v

def train_actor_critic(n_ep=500, gamma=0.99, ent_coef=0.01, val_coef=0.5):
    env = gym.make('CartPole-v1')
    model = ActorCritic(4, 2).to(device)
    opt = optim.Adam(model.parameters(), lr=3e-4)
    all_r = []
    for ep in tqdm(range(n_ep), desc='Training Actor-Critic'):
        s, _ = env.reset(); done = False
        a_losses, c_losses, ents = [], [], []
        ep_r = 0
        while not done:
            s_t = torch.FloatTensor(s).unsqueeze(0).to(device)
            logits, v = model(s_t)
            dist = Categorical(logits=logits); a = dist.sample()
            s2, r, te, tr, _ = env.step(a.item()); done = te or tr
            with torch.no_grad():
                _, v2 = model(torch.FloatTensor(s2).unsqueeze(0).to(device))
                v2 = torch.zeros(1,1).to(device) if done else v2

            # TODO: compute TD error delta = r + gamma*V(s') - V(s)
            delta = None   # <-- replace

            # TODO: actor loss  = -log_prob * delta.detach()
            #        critic loss = delta^2
            a_losses.append(None)  # <-- replace
            c_losses.append(None)  # <-- replace
            ents.append(dist.entropy())
            s = s2; ep_r += r

        # TODO: combine losses:  actor + val_coef*critic - ent_coef*entropy
        loss = None   # <-- replace

        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5); opt.step()
        all_r.append(ep_r)
        if (ep+1) % 100 == 0:
            print(f'Ep {ep+1:4d} | Mean-100: {np.mean(all_r[-100:]):.1f}')
    env.close(); return all_r, model


ac_rewards, ac_model = train_actor_critic()
plot_rewards(ac_rewards, 'Actor-Critic (TD) on CartPole-v1')


---
# Part 4 - Summary & Comparison

## Algorithm Overview

| Algorithm | Update | Bias | Variance | Stability |
|-----------|--------|------|----------|-----------|
| REINFORCE | Monte Carlo | Low | High | Low |
| REINFORCE + baseline | Monte Carlo + learned value baseline | Low | Medium | Medium |
| Actor-Critic (TD) | 1-step TD | High | Low | Medium |



In [ ]:
# Compare REINFORCE vs REINFORCE + baseline vs Actor-Critic
plt.figure(figsize=(9,3))
plt.plot(smooth(reinforce_rewards), label='REINFORCE', color='steelblue')
plt.plot(smooth(reinforce_bl_rewards), label='REINFORCE + baseline', color='darkgreen')
plt.plot(smooth(ac_rewards), label='Actor-Critic', color='darkorange')
plt.axhline(495, color='gray', ls='--', alpha=0.5, label='Solved')
plt.xlabel('Episode'); plt.ylabel('Smoothed Return')
plt.title('REINFORCE vs REINFORCE + baseline vs Actor-Critic'); plt.legend(); plt.tight_layout(); plt.show()  


---
# Further Reading

### Papers
1. Williams (1992) - REINFORCE: [link](https://link.springer.com/article/10.1007/BF00992696)
2. Sutton et al. (2000) - Policy Gradient Theorem: [link](https://proceedings.neurips.cc/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html)


### Books & Guides
- Sutton & Barto: [RL: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) (free PDF)
- [OpenAI Spinning Up](https://spinningup.openai.com/)

### Code
- [CleanRL](https://github.com/vwxyzjn/cleanrl) — single-file, readable RL implementations
- [Stable Baselines 3](https://github.com/DLR-RM/stable-baselines3) — production quality
